# group_analysis.ipynb

Evaluates the **decision-unit** grouping rule at the person level: how often the
people the rule glues together actually share an origin, and how often it splits
apart people who moved as one.

In [1]:
import numpy as np
import pandas as pd

In [2]:
year = 2018
path = "us/pums_2018_raw.csv"

In [ ]:
# Read only the columns this notebook uses -- 16 of the 300 in the IPUMS extract.
# Same convention as unify_pums.ipynb: adding a variable downstream means adding it
# here too, otherwise it KeyErrors rather than silently going missing.
USECOLS = [
    # identity / weights / household attributes (IPUMS is a pre-merged person+hh file)
    "CBSERIAL",
    "PERNUM",
    "PERWT",
    "HHWT",
    "NUMPREC",
    "GQ",
    # geography and the origin/destination outcome
    "STATEFIP",
    "PUMA",
    "MIGPLAC1",
    "MIGPUMA1",
    "MIGPUMANOW",
    # decision-unit construction
    "SUBFAM",
    "SFRELATE",
    "RELATE",
    "RELATED",
    "AGE",
    "PERWT",
]

df = pd.read_csv(path, usecols=USECOLS)
df.shape

(3214539, 16)

In [4]:
print(df["PERWT"].sum())

327167439.0


In [4]:
for col in df.columns:
    print(col)

CBSERIAL
NUMPREC
HHWT
STATEFIP
PUMA
GQ
PERNUM
PERWT
SUBFAM
SFRELATE
RELATE
RELATED
AGE
MIGPLAC1
MIGPUMA1
MIGPUMANOW


In [5]:
# origin is the MIGPLAC1 + MIGPUMA1 (migpuma geography)
df["ORIGIN"] = df["MIGPLAC1"].astype(int).astype(str).str.zfill(2) + df[
    "MIGPUMA1"
].astype(int).astype(str).str.zfill(5)

# chosen is the current location, STATEFIP + PUMA (puma geography)
df["CHOSEN"] = df["STATEFIP"].astype(int).astype(str).str.zfill(2) + df["PUMA"].astype(
    int
).astype(str).str.zfill(5)

# MIGPUMANOW is the current residence expressed in migpuma geography, so CHOSEN and
# ORIGIN can be compared directly -- no PUMA->MIGPUMA equivalency lookup needed
df["CHOSEN_MIGPUMA"] = df["STATEFIP"].astype(int).astype(str).str.zfill(2) + df[
    "MIGPUMANOW"
].astype(int).astype(str).str.zfill(5)

In [6]:
# people who stayed have MIGPLAC1 == MIGPUMA1 == 0; backfill their origin to the
# MIGPUMA they are currently in (chosen == origin)
df["ORIGIN"] = np.where(df["ORIGIN"] == "0000000", df["CHOSEN_MIGPUMA"], df["ORIGIN"])

# fill in the origin state with this backfill in place. Foreign origins get a
# MIGPLAC1 >= 100, which makes ORIGIN longer than 7 chars -- flagged as 999 so the
# contiguous-US filter below drops them.
df["ORIGIN_STATE"] = np.where(
    df["ORIGIN"].str.len() == 7, df["ORIGIN"].str[:2].astype(int), 999
)

# define STAY as not moving outside the MIGPUMA
df["STAY"] = df["CHOSEN_MIGPUMA"] == df["ORIGIN"]

In [7]:
# filtering to people who moved from places in the contiguous united states to other
# places in the contiguous united states (2 and 15 are Alaska and Hawaii)
mask = (
    # (df["ORIGIN_STATE"] <= 56)
    # & (~df["ORIGIN_STATE"].isin([2, 15]))
    # & (df["STATEFIP"] <= 56)
    # & (~df["STATEFIP"].isin([2, 15]))
    # institutionalized GQ (RELATED 1301) are outside of the context of this study
    # df["RELATE"] != 13
)
print(df.shape)
df = df.loc[mask].copy()
print(df.shape)

(3214539, 21)
(3214539, 21)


In [8]:
df["ORIGIN"].value_counts()

ORIGIN
0603700     102203
2500390      49638
1703400      41902
0400100      41174
4804600      36741
             ...  
33000001        75
35000001        73
52000001        71
62300001        66
62200001        61
Name: count, Length: 1038, dtype: int64

In [9]:
df["ORIGIN_STATE"].value_counts()

ORIGIN_STATE
6      377274
48     265623
12     199113
36     197113
42     128757
17     126968
39     118907
37     101294
13      99777
26      99181
34      88714
51      84057
53      75492
25      69599
4       68702
18      67427
47      67323
29      62272
24      59648
55      59634
27      55797
8       55563
45      49125
1       47512
21      45229
22      43570
41      41714
40      37582
9       36349
19      32195
49      31177
5       30415
20      29519
28      29075
32      28499
31      19450
35      19155
54      18106
999     17841
16      16517
15      14390
33      13648
23      13117
44      10325
30      10287
46       9075
10       9056
38       7882
2        6858
11       6510
50       6388
56       5738
Name: count, dtype: int64

In [10]:
df["CHOSEN"].value_counts()

CHOSEN
0102500    4550
5310200    4083
5500700    4082
5500100    3958
1200500    3421
           ... 
2701503     580
5541001     575
2701403     566
2701402     523
4203207     509
Name: count, Length: 2351, dtype: int64

In [11]:
df["STAY"].value_counts()

STAY
True     3025964
False     188575
Name: count, dtype: int64

In [12]:
def agreement(sub, name, codes=None, col="RELATED"):
    """Share of households where all listed members agree, on ORIGIN and on STAY."""
    if codes is not None:
        sub = sub[sub[col].isin(codes)]
        n = sub.groupby("CBSERIAL")[col].nunique()
        sub = sub[sub["CBSERIAL"].isin(n[n == len(codes)].index)]
    size = sub.groupby("CBSERIAL").size()
    sub = sub[sub["CBSERIAL"].isin(size[size >= 2].index)]
    g = sub.groupby("CBSERIAL")[["ORIGIN", "STAY"]].nunique()
    # this is on nunique
    a_o = (g["ORIGIN"] == 1).mean()
    a_s = (g["STAY"] == 1).mean()
    print(
        f"{name:34s} n={len(g):>8,}  origin agree={a_o:.4f}  stay agree={a_s:.4f}  gap={a_s - a_o:.4f}"
    )


# a_o (ORIGIN agreement): everybody reports the same origin MIGPUMA. Splits into
#   households that never left, and households that moved as a unit. The latter
#   is the population the joint-decision model is about.
#
# a_s (STAY agreement): everybody all-stayed or all-moved. Weaker than a_o, since
#   STAY collapses every distinct origin into one "moved" bucket. In practice
#   a_s is ~"everyone stayed": 95% of persons have STAY=True, and the all-moved
#   branch is almost entirely already inside a_o.
#
# gap = a_s - a_o: all-moved households arriving from DIFFERENT origins
#   (in-migrants converging to form a household). A lower bound on merging --
#   the commoner pattern is a joiner moving into a household that never left,
#   which disagrees on STAY and so falls outside the gap.
#
# the households outside origin agree did things like have people move in externally with a person who stayed
#
# IPUMS relationship codes used below (RELATED unless noted):
#   101 householder   201 spouse    1114 unmarried partner   1115 housemate/roommate
#   RELATE 3 own child (301/302/303 = bio/adopted/step)      RELATE 10 other relative
agreement(df, "all multi-person households")
agreement(df, "householder + spouse", [101, 201])
agreement(df, "householder + unmarried partner", [101, 1114])
agreement(df, "householder + own child", [1, 3], col="RELATE")
agreement(df, "householder + roommate/boarder", [101, 1115])
agreement(df, "householder + other relative", [1, 10], col="RELATE")

# --- adults only, for comparison with the modeled sample ----------------------
print()
agreement(df[df["AGE"] >= 18], "adults only: all multi-person")
agreement(df[df["AGE"] >= 18], "adults only: hh + spouse", [101, 201])

all multi-person households        n= 910,652  origin agree=0.9535  stay agree=0.9575  gap=0.0040
householder + spouse               n= 640,504  origin agree=0.9921  stay agree=0.9931  gap=0.0011
householder + unmarried partner    n=  73,373  origin agree=0.9184  stay agree=0.9325  gap=0.0141
householder + own child            n= 455,440  origin agree=0.9695  stay agree=0.9707  gap=0.0012
householder + roommate/boarder     n=  30,210  origin agree=0.7902  stay agree=0.8243  gap=0.0341
householder + other relative       n=  21,448  origin agree=0.8881  stay agree=0.8932  gap=0.0052

adults only: all multi-person      n= 861,919  origin agree=0.9559  stay agree=0.9601  gap=0.0042
adults only: hh + spouse           n= 640,426  origin agree=0.9921  stay agree=0.9931  gap=0.0011


In [13]:
multi = df.groupby("CBSERIAL").filter(lambda g: len(g) >= 2)
g = multi.groupby("CBSERIAL").agg(
    n_origin=("ORIGIN", "nunique"),
    n_stay=("STAY", "nunique"),
    all_moved=("STAY", lambda s: (~s.astype(bool)).all()),
)

coherent = g["n_origin"] == 1
merge_converge = (~coherent) & g[
    "all_moved"
]  # the "gap": all in-migrants, diff origins
merge_join = (~coherent) & (g["n_stay"] == 2)  # incumbent + joiner

for name, mask in [
    ("coherent unit", coherent),
    ("merger: all arrived, diff origins", merge_converge),
    ("merger: joined existing household", merge_join),
]:
    print(f"{name:38s} {mask.sum():>8,}  {mask.mean():.4f}")

coherent unit                           868,294  0.9535
merger: all arrived, diff origins         3,617  0.0040
merger: joined existing household        38,741  0.0425


In [14]:
df["RELATED"].value_counts()

RELATED
101     1257501
301      759301
201      640504
1270      79139
1301      74336
1114      73373
901       69104
1115      41715
303       34031
501       33901
1260      32635
1001      29836
701       29592
302       19528
401       11923
1241      11777
601        9143
801        4791
1242       2409
Name: count, dtype: int64

In [15]:
# NUMPREC is the IPUMS household size (raw PUMS NP)
df["NUMPREC"].value_counts()

NUMPREC
2     925302
4     598584
3     563019
1     500324
5     338765
6     156840
7      66290
8      31648
9      15030
10      8410
11      4510
12      2904
13      1066
14       616
15       465
20       280
17       238
16       176
18        72
Name: count, dtype: int64

In [16]:
# GQ 1/2 = households, 3/4/5 = group quarters
adults = df[df["GQ"].isin([1, 2]) & (df["AGE"] >= 18)].copy()
adults = adults[adults.groupby("CBSERIAL")["AGE"].transform("size") > 1]

g = adults.groupby("CBSERIAL")
ref = adults[adults["RELATE"] == 1].set_index("CBSERIAL")
w = ref["PERWT"]


def report(label, mask):
    m = mask.reindex(ref.index).fillna(False)
    print(f"{label:<58} {m.mean():>7.2%} {w[m].sum() / w.sum():>8.2%}")


print(f"Multi-adult households: {len(ref):,}")
print(f"Mean adults per household: {g.size().reindex(ref.index).mean():.2f}\n")

# these need EDUCD / RACE / HISPAN / INDNAICS pulled into USECOLS first
# report(
#     "Any adult BA+ but reference person not",
#     (g["EDU_HAS_DEGREE"].max() == 1) & (ref["EDU_HAS_DEGREE"] == 0),
# )
# report("Household mixed on BA+", g["EDU_HAS_DEGREE"].nunique() > 1)
# report("More than one race group", g["RACE_ETHNICITY"].nunique() > 1)
# report("Age spread > 16 years", (g["AGE"].max() - g["AGE"].min()) > 16)

# # industry -- IPUMS INDNAICS, grouped however create_estdata groups it
# emp = adults[adults["INDNAICS"].notna()]
# by_hh = emp.groupby("CBSERIAL")["INDNAICS"]
# report("More than one industry among employed", by_hh.nunique() > 1)
# report(
#     "Employed adult differs from reference person",
#     by_hh.apply(lambda s: (s != ref["INDNAICS"].get(s.name)).any()),
# )

Multi-adult households: 861,811
Mean adults per household: 2.34



In [17]:
# Household movement concordance: for each household (CBSERIAL), do all members
# share the same STAY status (all moved together vs. a mixed household)?
household = df.groupby("CBSERIAL").agg(
    n_members=("STAY", "size"),
    n_unique_stay=("STAY", "nunique"),
    # IPUMS carries the household weight on every person row, so unlike the raw-PUMS
    # version this is a real HHWT rather than the first member's person weight
    weight=("HHWT", "first"),
)
household["unanimous"] = household["n_unique_stay"] == 1

n_households = len(household)
print(f"Total households: {n_households}")
print(
    f"Unanimous (everyone moved or everyone stayed): {household['unanimous'].mean():.2%}"
)
print(f"Mixed (some moved, some stayed): {(~household['unanimous']).mean():.2%}")

weighted_unanimous = (
    household.loc[household["unanimous"], "weight"].sum() / household["weight"].sum()
)
print(f"Weighted share of unanimous households: {weighted_unanimous:.2%}")

# single-person households are trivially unanimous, so also look at multi-person only
multi_person = household[household["n_members"] > 1]
print(
    f"\nAmong multi-person households: {multi_person['unanimous'].mean():.2%} unanimous"
)

# among unanimous households, how many moved together vs. stayed together
unanimous_ids = household[household["unanimous"]].index
unanimous_status = (
    df[df["CBSERIAL"].isin(unanimous_ids)]
    .drop_duplicates("CBSERIAL")
    .set_index("CBSERIAL")["STAY"]
)
print("\nAmong unanimous households, share where everyone moved vs everyone stayed:")
print(unanimous_status.value_counts(normalize=True))

Total households: 1410976
Unanimous (everyone moved or everyone stayed): 97.25%
Mixed (some moved, some stayed): 2.75%
Weighted share of unanimous households: 97.07%

Among multi-person households: 95.75% unanimous

Among unanimous households, share where everyone moved vs everyone stayed:
STAY
True     0.935798
False    0.064202
Name: proportion, dtype: float64


In [18]:
df[~df.CBSERIAL.map(household["unanimous"]) & ~df["STAY"].astype(bool)][
    "RELATED"
].value_counts()

RELATED
301     12129
101     10517
201      5058
1115     4539
1114     3505
1260     3199
901      2974
1001     2331
501      1699
701      1617
303      1328
1241     1172
401      1002
601       783
801       481
302       385
1242      286
Name: count, dtype: int64

In [19]:
df[~df.CBSERIAL.map(household["unanimous"]) & df["STAY"].astype(bool)][
    "RELATED"
].value_counts()

RELATED
101     28224
301     19880
201     12539
1115     5316
901      3614
1114     2965
1260     2629
1001     1868
501      1488
701      1454
303      1136
1241      969
302       661
401       622
601       393
801       289
1242      223
Name: count, dtype: int64

In [20]:
# SUBFAM is the IPUMS subfamily number (raw PUMS SFN); SFRELATE is the member's role
# within it. SUBFAM is occasionally nonzero while SFRELATE is 0, which is why the UNIT
# rule below requires both.
print(df["SUBFAM"].value_counts())
print(df["SFRELATE"].value_counts())

SUBFAM
0    3108761
1     103016
2       2627
3        127
4          5
5          3
Name: count, dtype: int64
SFRELATE
0    3108896
3      48067
1      41095
2      16481
Name: count, dtype: int64


In [21]:
"""Baseline (unit = whole household), from the raw-ACS-PUMS run of this notebook.
Kept for reference -- the numbers below are NOT from the IPUMS data loaded above,
they are the household-as-unit comparison point the rule is scored against.

persons            : 2,386,856
households         : 1,246,447
decision units     : 1,283,126
reduction vs person: 46.2%
unit size dist     : {1: 428073, 2: 673385, 3: 131399, 4: 38299, 5: 9004, 6: 2030, 7: 553, 8: 198} -> this excludes children under 18
multi-person units : 855,053  share agreeing on ORIGIN: 0.9639

misattributed people (tie-free) : 33,460  (0.0140 of all)
flagged, all people                33,460  (0.0140 of 2,386,856)  weighted=4,024,287
flagged, in multi-person units     33,460  (0.0171 of 1,958,783)  weighted=4,024,287

misattributed movers in multi-person units: 27,009 (0.3196 of 84,510)  weighted=3,233,428

RELP of misattributed movers (2-person units excluded -- tie-break there
is arbitrary, so 'which' member is blamed carries no meaning):
RELP
2     4613
12    2987
15    1566
0     1270
10    1029
5      746
11     644
6      628
7      543
4      488
Name: count, dtype: int64

1,553,274 within-household pairs
people separated from a same-origin housemate: 121,008
  of which involve a mover: 1,375
"""

"Baseline (unit = whole household), from the raw-ACS-PUMS run of this notebook.\nKept for reference -- the numbers below are NOT from the IPUMS data loaded above,\nthey are the household-as-unit comparison point the rule is scored against.\n\npersons            : 2,386,856\nhouseholds         : 1,246,447\ndecision units     : 1,283,126\nreduction vs person: 46.2%\nunit size dist     : {1: 428073, 2: 673385, 3: 131399, 4: 38299, 5: 9004, 6: 2030, 7: 553, 8: 198} -> this excludes children under 18\nmulti-person units : 855,053  share agreeing on ORIGIN: 0.9639\n\nmisattributed people (tie-free) : 33,460  (0.0140 of all)\nflagged, all people                33,460  (0.0140 of 2,386,856)  weighted=4,024,287\nflagged, in multi-person units     33,460  (0.0171 of 1,958,783)  weighted=4,024,287\n\nmisattributed movers in multi-person units: 27,009 (0.3196 of 84,510)  weighted=3,233,428\n\nRELP of misattributed movers (2-person units excluded -- tie-break there\nis arbitrary, so 'which' member 

In [22]:
# The decision-unit rule under evaluation, ported to IPUMS codes:
#   PRIMARY  RELATE 1/2   = householder, spouse
#   CHILDREN RELATE 3/9   = own child (bio/adopted/step), grandchild
#            RELATED 1242 = foster child
#
# NB this is the rule this notebook has always scored, NOT verbatim the one
# unify_pums.ipynb ships. unify_pums differs in two ways: it folds unmarried
# partners (RELATED 1114) into the primary unit, and it takes related children at
# AGE < 18 (RELATE < 11) rather than own children/grandchildren at AGE < 25. Change
# the clauses below to score that variant instead.
PRIMARY = {1, 2}
df["UNIT"] = np.where(
    # sometimes subfam != 0 while sfrelate == 0
    (df.SUBFAM != 0) & (df.SFRELATE != 0),
    # subfamily → its own unit
    df.CBSERIAL.astype(str) + "_SF" + df.SUBFAM.astype(str),
    np.where(
        # count primary
        df.RELATE.isin(PRIMARY)
        # count related children don't count unrelated children, count ofster children
        | ((df.AGE < 18) & ((df.RELATE < 11) | (df.RELATED.isin([1242]))))
        # unmarried partner
        | df.RELATED.isin([1114]),
        # primary family, non-subfamily or child with age < 18
        df.CBSERIAL.astype(str) + "_P",
        # treat everyone else as singletons, individual decision units
        df.CBSERIAL.astype(str) + "_I" + df.PERNUM.astype(int).astype(str),
    ),
)

In [23]:
# working frame for the whole evaluation section (drops GQ since there truly aren't
# any aggregate decision units there)
d = df[
    [
        "CBSERIAL",
        "PERNUM",
        "SUBFAM",
        "SFRELATE",
        "RELATE",
        "RELATED",
        "ORIGIN",
        "STAY",
        "UNIT",
        "PERWT",
    ]
].copy()
d["PID"] = d["CBSERIAL"].astype(str) + "_" + d["PERNUM"].astype(int).astype(str)

sizes = d.groupby("UNIT").size()
print(f"persons            : {len(d):,}")
print(f"households         : {d['CBSERIAL'].nunique():,}")
print(f"decision units     : {d['UNIT'].nunique():,}")
print(f"reduction vs person: {100 * (1 - d['UNIT'].nunique() / len(d)):.1f}%")
print(f"unit size dist     : {sizes.value_counts().sort_index().head(8).to_dict()}")

u = d.groupby("UNIT")["ORIGIN"].nunique()
multi_u = sizes[sizes >= 2].index
print(
    f"multi-person units : {len(multi_u):,}  "
    f"share agreeing on ORIGIN: {(u.loc[multi_u] == 1).mean():.4f}"
)

# ----------------------------------------------------------------------------
# persons            people left after dropping GQ and under-18s. The estimation
#                    sample if you modelled individuals.
# households         distinct CBSERIALs. Not the decision unit -- shown only so the
#                    next line can be compared against it.
# decision units     rows you would actually estimate on. Always >= households,
#                    beacause the logic splits household units into 1 or more decision units
# reduction vs person how much the grouping shrinks the sample. Pure cost: it is the
#                    power you trade away for a more defensible decision unit.
#                    Compare against the baseline (~46% for household-as-unit); a
#                    SMALLER reduction means a stricter, more conservative rule.
# unit size dist     units of size 1 are people modelled individually anyway -- they
#                    cannot be misattributed and dilute every rate computed over
#                    "all people". Watch how many units are size 1 (excluding children)
# share agreeing     of multi-person units, the fraction where every member reports
#                    the same ORIGIN.

persons            : 3,214,539
households         : 1,410,976
decision units     : 1,845,792
reduction vs person: 42.6%
unit size dist     : {1: 1020686, 2: 530282, 3: 128897, 4: 108293, 5: 40535, 6: 12019, 7: 3247, 8: 1122}
multi-person units : 825,106  share agreeing on ORIGIN: 0.9791


### Reporting at the person level

Everything below counts **people**, not pairs.

There are exactly two ways the grouping rule can be wrong, and they are different
kinds of wrong. Both are measured against the same metric -- whether two people
report the same `ORIGIN` (the MIGPUMA they lived in a year ago), which indicates they are
(probably) a single deiciosn unit

---

#### misattributed -- the rule glued together people who do not belong together

A decision unit is a claim: *"these people faced one origin and jointly picked one
destination."* A member is **misattributed** when their own `ORIGIN` is not the
unit's origin -- the claim is false for them.

> A unit of three in Chicago. Two members lived in Phoenix a year ago; the third
> was already in Chicago and never moved. The unit's origin is Phoenix. That third
> person is misattributed.

When you estimate, that person contributes an observation reading *"faced origin
Phoenix, chose Chicago"* -- a choice situation they were never in. It is a
**fabricated observation**, and the MNL cannot tell it apart from a real one. This
is the dangerous error: it does not just add noise, it biases coefficients, because
the fake observations are not randomly distributed (they concentrate in households
that took in a mover).

The unit's origin is the **reference person's** origin -- not a majority vote --
because that is the origin `unify_pums.ipynb` writes onto the unit and the model
estimates on. So the flag marks everyone who disagrees with the reference person,
the reference person is never flagged, and in the example above the two Phoenix
arrivals are flagged if the never-moved member is the householder. The count still
includes both movers and stayers.

---

#### separated -- the rule split apart people who do belong together

Two people live together and report the same `ORIGIN`, but the rule assigned them to
different units.

> A householder and their 30-year-old sibling, both of whom moved from Denver to
> Atlanta together last year. `RELATE=7` (sibling) is not in `PRIMARY`, so the sibling
> becomes a singleton and the two are modelled separately.

The model now treats one joint decision as **two independent decisions** -- exactly
the assumption this whole exercise set out to escape. Nothing is fabricated; the
observations are real. You have simply failed to aggregate, so those people revert
to the individual-level model you started with.

---

#### Why they are never combined into one score

| | misattributed | separated |
|---|---|---|
| what happens | invents a choice nobody made | records a real choice, twice |
| statistical cost | **bias** -- wrong coefficients | **inefficiency** -- overstated sample, understated SEs |
| fallback if wrong | corrupted data | the old individual model |

Misattribution is strictly worse. A rule that removes 100 misattributions at the
cost of 100 separations is a good trade.

Every rule faces this tension: grouping more people catches more genuine joint
moves (fewer separated) but inevitably sweeps in people who do not belong (more
misattributed). Household-as-unit sits at one extreme; modelling everyone
individually sits at the other.

In [24]:
# MISATTRIBUTED = people the rule glued into a unit they do not belong to.
#
# A unit asserts "these people shared one origin and jointly chose one destination".
# A member is misattributed when their own ORIGIN is not the unit's origin, so that
# assertion is false for them. At estimation they contribute a choice situation they
# were never in -- a fabricated observation the MNL treats as real. This is the
# error that BIASES coefficients, which is why it is the number to minimise.
#
# The unit's origin is the REFERENCE PERSON's origin -- no plurality vote and no
# tie-break. This matches how unify_pums.ipynb actually builds a unit: every _REF
# covariate, and BASE_COLS (ORIGIN, CHOSEN, STAY) with them, is read off REF_INDEX,
# so the reference person's origin IS the origin the model estimates on. Scoring
# against a plurality would grade a unit on an origin the pipeline never assigns it.
#
# Reference person = SFRELATE == 1 within a subfamily unit, else RELATE == 1
# (householder), else the earliest-listed member -- identical to unify_pums's
# _REF_PRIORITY / REF_INDEX. Consequence: the reference person can never be flagged,
# and in a unit where two members moved in together and the householder never left,
# it is now the two MOVERS who are flagged, not the incumbent. Under the old
# plurality rule the minority was blamed and that was the other way round.
d["_REF_PRIORITY"] = np.where((d["SFRELATE"] == 1) | (d["RELATE"] == 1), 0, 1)
ref_index = d.groupby("UNIT", sort=False)["_REF_PRIORITY"].idxmin()
unit_origin = d.loc[ref_index.values, "ORIGIN"]
unit_origin.index = ref_index.index

d["misattributed"] = d["ORIGIN"] != d["UNIT"].map(unit_origin)

grouped = d[d["UNIT"].map(sizes) > 1]  # singletons cannot be misattributed
for label, sub in [("all people", d), ("in multi-person units", grouped)]:
    n = int(sub["misattributed"].sum())
    w = sub.loc[sub["misattributed"], "PERWT"].sum()
    print(
        f"flagged, {label:<22} {n:>9,}  ({n / len(sub):.4f} of {len(sub):,})"
        f"  weighted={w:,.0f}"
    )

# ----------------------------------------------------------------------------
# Both printed lines count THE SAME PEOPLE; they differ only in denominator.
#
# "flagged, all people"       everyone whose ORIGIN differs from their unit
#                    reference person's.
#
# "flagged, in multi-person"  same people, denominator restricted to units of 2+.
#                    Singletons are their own reference person and so can never be
#                    flagged; including them only dilutes the rate. This is the
#                    honest denominator of the two.
#
# weighted=          PERWT-weighted misattribution count, i.e. the national population estimate
#
# Note this count is no longer symmetric the way the old plurality "tie-free" count
# was: a 2-person unit that disagrees still contributes exactly 1, but WHICH member
# is named is now determined rather than arbitrary, so the RELATED breakdowns below
# are meaningful for 2-person units too.

flagged, all people                20,512  (0.0064 of 3,214,539)  weighted=2,188,532
flagged, in multi-person units     20,512  (0.0093 of 2,193,853)  weighted=2,188,532


In [25]:
# movers only (people)
mv = grouped[~grouped["STAY"].astype(bool)]
n = int(mv["misattributed"].sum())
print(
    f"misattributed movers in multi-person units: {n:,} "
    f"({n / len(mv):.4f} of {len(mv):,})  "
    f"weighted={mv.loc[mv['misattributed'], 'PERWT'].sum():,.0f}"
)

print("\nRELATED of misattributed movers, all multi-person units:")
print(mv.loc[mv["misattributed"], "RELATED"].value_counts().head(10))

# units of 3+ only, for comparison with the old plurality-scored numbers, where
# 2-person units had to be dropped because the tie-break decided who got blamed
print("\nsame, units of 3+ only:")
big = mv[mv["UNIT"].map(sizes) > 2]
print(big.loc[big["misattributed"], "RELATED"].value_counts().head(10))

# ----------------------------------------------------------------------------
# "misattributed movers"  a subset of the total above, restricted to STAY == False.
#                    The remainder (total - this) are misattributed stayers: people
#                    outnumbered in their own unit by members who arrived from
#                    elsewhere.
#
# the rate           misattributed movers / movers in multi-person units. This is
#                    the fraction of the identifying sample carrying a fabricated
#                    destination choice -- a mover credited with a move they did not
#                    make. Compare it against the baseline rate (~0.32 for
#                    household-as-unit); the drop is what the rule buys you.
#
# RELATED breakdown  which relationship types the residual error sits in. The
#                    reference person is never flagged, so this reads as "which kind
#                    of housemate is the rule wrongly attaching to a householder":
#                      301/302/303 (own children) -- adults living with parents. Since
#                        under-18s are excluded these are grown children who moved
#                        in or out independently, and are untouched by any PRIMARY
#                        variant tested.
#                      101 (householder)     -- only possible inside a subfamily unit,
#                        where the subfamily head (SFRELATE 1) outranks them.
#                      1115/1260/1001/701    -- housemates, other nonrelatives, other
#                        relatives, siblings. If these are large the rule is not
#                        splitting off the people it was designed to split off.

misattributed movers in multi-person units: 11,989 (0.1296 of 92,533)  weighted=1,266,173

RELATED of misattributed movers, all multi-person units:
RELATED
1114    4171
201     3322
301     1440
901     1172
1001     552
303      468
1242     274
401      158
302      102
1260      93
Name: count, dtype: int64

same, units of 3+ only:
RELATED
301     1271
201      987
1114     891
901      853
303      461
1001     422
1242     266
302       93
401       64
701       49
Name: count, dtype: int64


In [26]:
# SEPARATED = people the rule split apart who do belong together.
#
# Two people live in the same household and report the same ORIGIN -- they plausibly
# moved as one -- but the rule put them in different units. Nothing is fabricated;
# both observations are real. The model just treats one joint decision as two
# independent ones, which is the assumption this exercise set out to escape. Those
# people revert to the individual-level model.
#
# Cost is INEFFICIENCY, not bias: the sample looks larger than it is, so standard
# errors come out too small. Strictly less damaging than misattribution, which is
# why the two are reported separately and never averaged together.

# People split away from a housemate they actually share an ORIGIN with.
# This one is inherently relational, so it needs the within-household pair table to determine the number of offending pairs.
pairs = d.merge(d, on="CBSERIAL", suffixes=("_a", "_b"))
pairs = pairs[pairs["PERNUM_a"] < pairs["PERNUM_b"]]
pairs["same_origin"] = pairs["ORIGIN_a"] == pairs["ORIGIN_b"]
pairs["same_unit"] = pairs["UNIT_a"] == pairs["UNIT_b"]
print(f"{len(pairs):,} within-household pairs")

split_pairs = pairs[~pairs["same_unit"] & pairs["same_origin"]]
separated = pd.unique(
    pd.concat([split_pairs["PID_a"], split_pairs["PID_b"]], ignore_index=True)
)
print(f"people separated from a same-origin housemate: {len(separated):,}")

mover_split = split_pairs[
    ~split_pairs["STAY_a"].astype(bool) | ~split_pairs["STAY_b"].astype(bool)
]
sep_mv = pd.unique(
    pd.concat([mover_split["PID_a"], mover_split["PID_b"]], ignore_index=True)
)
print(f"  of which involve a mover: {len(sep_mv):,}")

# ----------------------------------------------------------------------------
# This is the COST side. Misattribution puts fabricated choices into the data
# (bias); separation only forgoes aggregation, treating potentially a single decision unit as 2+
# decision units. This is arguably the lesser evil compared to misattribution.
#
# "within-household pairs"   size of the pair table. Diagnostic only.
#
# "people separated"         Pretty much just quantifies how much the rule separates people.
#                            Overwhelmingly dominated by stayers who have the same origin as they didn't move.
#
# "of which involve a mover" THIS is the real cost.
#
# Counted per PERSON, not per pair: one misplaced person in a 4-person unit would
# otherwise generate three bad pairs and triple-count the problem.

3,459,452 within-household pairs
people separated from a same-origin housemate: 1,028,228
  of which involve a mover: 17,295


In [27]:
two = mv[mv["UNIT"].map(sizes) == 2]
pair_relate = (
    d[d["UNIT"].isin(two["UNIT"])]
    .groupby("UNIT")["RELATED"]
    .agg(lambda s: tuple(sorted(s)))
)
print(
    pair_relate[pair_relate.index.isin(two.loc[two["misattributed"], "UNIT"])]
    .value_counts()
    .head(10)
)

RELATED
(101, 1114)     3280
(101, 201)      2335
(301, 901)       215
(101, 301)       135
(301, 401)       109
(101, 901)        82
(1260, 1260)      54
(101, 1001)       46
(1115, 1115)      36
(1001, 1001)      36
Name: count, dtype: int64
